# Semester Assignment Beginnings

**This is where I will mess around with ideas on what I can do for the semester assignment**

### Baseball

Train multiple regression models (with different variables in different configurations) on a few players' stats and try to estimate how they will perform in the future (with past data so you can check the results). Run uncertianty analyses on the regression models with some simulations to see how accurate the results are. 

Explain basic baseball rules to understand what the different variables mean (will add free words to the assignment).

Visualize the data, make some initial deductions and preform some exploratory analyses to get a deeper insight of the data. 

In [121]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

pd.set_option('display.max_columns', None) 

In [122]:
# Limiting what columns are inluded to only get the relevant information.
batting = pd.read_csv('Data/batting.csv', 
                      usecols = ['player_id', 'year', 'stint', 'team_id', 'league_id', 'g', 'ab', 'r', 'h', 'double', 'triple', 'hr', 'rbi', 'sb', 'cs', 'bb', 'so', 'hbp', 'sh', 'sf'])
fielding = pd.read_csv('Data/fielding.csv', 
                       usecols = ['player_id', 'year', 'stint', 'team_id', 'league_id', 'pos', 'g', 'po', 'a', 'e', 'dp'])
pitching = pd.read_csv('Data/pitching.csv', 
                       usecols = ['player_id', 'year', 'stint', 'team_id', 'league_id', 'w', 'l', 'g', 'sho', 'sv', 'ipouts', 'h', 'er', 'hr', 'bb', 'so', 'era', 'hbp', 'bk', 'bfp'])
player = pd.read_csv('Data/player.csv', 
                     usecols = ['player_id', 'birth_year', 'name_first', 'name_last', 
                                'weight', 'height', 'bats', 'throws'])
salary = pd.read_csv('Data/salary.csv', 
                     usecols = ['year', 'player_id', 'team_id', 'league_id', 'salary'])
team = pd.read_csv('Data/team.csv', 
                   usecols = ['year', 'league_id', 'team_id', 'franchise_id', 'div_id', 'rank', 'g', 'w', 'l', 'div_win', 'wc_win', 'lg_win', 'ws_win', 'r', 'ab', 'h', 'double', 'triple', 'hr', 'bb', 'so', 'sb', 'cs', 'hbp', 'sf', 'era', 'sho', 'sv', 'ipouts', 'e'])
team_names = pd.read_csv('Data/team.csv', usecols = ['team_id', 'name'])
cpi = pd.read_csv('Data/cpi.csv')

In [123]:
batting = batting[(batting['year'] >= 1876) & (batting['league_id'].isin(['NL', 'AL']))]
fielding = fielding[(fielding['year'] >= 1876) & (fielding['league_id'].isin(['NL', 'AL']))]
pitching = pitching[(pitching['year'] >= 1876) & (pitching['league_id'].isin(['NL', 'AL']))]
team = team[(team['year'] >= 1876) & (team['league_id'].isin(['NL', 'AL']))]

In [124]:
player['weight'] = (player['weight'] * 0.453592).round() # Converting pounds to kg, rounding to nearest integer.
player['height'] = (player['height'] * 2.54).round() / 100 # Converting inches to cm, rounding to nearest integer.
player['bmi'] = (player['weight'] / (player['height'] ** 2)).round(1)

In [125]:
team_names = team_names.sort_values('team_id')

In [126]:
cpi['cpi'] = cpi[['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                  'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']].mean(axis = 1)
cpi = cpi[(cpi['Year'] >= 1985) & (cpi['Year'] <= 2015)]
cpi['Year'] = cpi['Year'].astype(int)
cpi = cpi.rename({'Year': 'year'}, axis = 1)

In [127]:
salary = pd.merge(salary, cpi[['year', 'cpi']], on = 'year', how = 'inner')
salary['adj_salary'] = (salary['salary'] / salary['cpi']) * salary.loc[salary['year'] == 2015, 'cpi'].values[0]

In [173]:
batting.loc[batting['player_id'] == 'zimmejo02']

,player_id,year,stint,team_id,league_id,g,ab,r,h,double,triple,hr,rbi,sb,cs,bb,so,hbp,sh,sf,adj_salary,weight,height,bmi,bats,throws,birth_year,age
89268,zimmejo02,2009,1,WAS,NL,16,27.0,1.0,4.0,0.0,0.0,0.0,2.0,0.0,0.0,1.0,11.0,0.0,6.0,0.0,NaN,102.0,1.88,28.9,R,R,1986.0,23.0
90624,zimmejo02,2010,1,WAS,NL,7,10.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,NaN,102.0,1.88,28.9,R,R,1986.0,24.0
92014,zimmejo02,2011,1,WAS,NL,29,43.0,1.0,9.0,1.0,0.0,0.0,3.0,0.0,0.0,2.0,12.0,0.0,11.0,0.0,4.372829e+05,102.0,1.88,28.9,R,R,1986.0,25.0
93422,zimmejo02,2012,1,WAS,NL,32,57.0,5.0,11.0,2.0,0.0,1.0,4.0,0.0,0.0,4.0,13.0,0.0,5.0,0.0,2.374362e+06,102.0,1.88,28.9,R,R,1986.0,26.0
94830,zimmejo02,2013,1,WAS,NL,32,65.0,4.0,8.0,1.0,0.0,0.0,2.0,0.0,0.0,1.0,20.0,0.0,6.0,1.0,5.443238e+06,102.0,1.88,28.9,R,R,1986.0,27.0
96266,zimmejo02,2014,1,WAS,NL,32,55.0,3.0,10.0,1.0,0.0,0.0,1.0,0.0,0.0,2.0,21.0,0.0,9.0,1.0,7.508897e+06,102.0,1.88,28.9,R,R,1986.0,28.0
97749,zimmejo02,2015,1,WAS,NL,33,63.0,4.0,10.0,1.0,0.0,0.0,3.0,0.0,0.0,0.0,18.0,0.0,6.0,0.0,1.650000e+07,102.0,1.88,28.9,R,R,1986.0,29.0


In [172]:
fielding.loc[fielding['player_id'] == 'zimmejo02']

,player_id,year,stint,team_id,league_id,pos,g,po,a,e,dp,adj_salary,weight,height,bmi,bats,throws,birth_year
149061,zimmejo02,2009,1,WAS,NL,P,16,7.0,16.0,1.0,1.0,NaN,102.0,1.88,28.9,R,R,1986.0
151375,zimmejo02,2010,1,WAS,NL,P,7,1.0,2.0,2.0,0.0,NaN,102.0,1.88,28.9,R,R,1986.0
153757,zimmejo02,2011,1,WAS,NL,P,26,18.0,19.0,0.0,1.0,4.372829e+05,102.0,1.88,28.9,R,R,1986.0
156219,zimmejo02,2012,1,WAS,NL,P,32,12.0,28.0,2.0,2.0,2.374362e+06,102.0,1.88,28.9,R,R,1986.0
158631,zimmejo02,2013,1,WAS,NL,P,32,12.0,28.0,2.0,1.0,5.443238e+06,102.0,1.88,28.9,R,R,1986.0
161116,zimmejo02,2014,1,WAS,NL,P,32,13.0,19.0,1.0,0.0,7.508897e+06,102.0,1.88,28.9,R,R,1986.0
161934,zimmejo02,2015,1,WAS,NL,P,33,16.0,29.0,2.0,1.0,1.650000e+07,102.0,1.88,28.9,R,R,1986.0


In [171]:
pitching.loc[pitching['player_id'] == 'zimmejo02']

,player_id,year,stint,team_id,league_id,w,l,g,sho,sv,ipouts,h,er,hr,bb,so,era,hbp,bk,bfp,adj_salary,weight,height,bmi,bats,throws,birth_year
38628,zimmejo02,2009,1,WAS,NL,3,5,16,0,0,274.0,95,47,10,29,92,4.63,4.0,0,391.0,NaN,102.0,1.88,28.9,R,R,1986.0
39312,zimmejo02,2010,1,WAS,NL,1,2,7,0,0,93.0,31,17,8,10,27,4.94,2.0,0,135.0,NaN,102.0,1.88,28.9,R,R,1986.0
40020,zimmejo02,2011,1,WAS,NL,8,11,26,0,0,484.0,154,57,12,31,124,3.18,7.0,1,662.0,4.372829e+05,102.0,1.88,28.9,R,R,1986.0
40742,zimmejo02,2012,1,WAS,NL,12,8,32,0,0,587.0,186,64,18,43,153,2.94,8.0,0,805.0,2.374362e+06,102.0,1.88,28.9,R,R,1986.0
41468,zimmejo02,2013,1,WAS,NL,19,9,32,2,0,640.0,192,77,19,40,161,3.25,7.0,0,865.0,5.443238e+06,102.0,1.88,28.9,R,R,1986.0
42215,zimmejo02,2014,1,WAS,NL,14,5,32,2,0,599.0,185,59,13,29,182,2.66,6.0,0,800.0,7.508897e+06,102.0,1.88,28.9,R,R,1986.0
43022,zimmejo02,2015,1,WAS,NL,13,10,33,0,0,605.0,204,82,24,39,164,3.66,8.0,1,831.0,1.650000e+07,102.0,1.88,28.9,R,R,1986.0


In [175]:
team_names.loc[team_names['team_id'] == 'WAS']

,team_id,name
290,WAS,Washington Senators
2774,WAS,Washington Nationals
2714,WAS,Washington Nationals
2504,WAS,Washington Nationals
2594,WAS,Washington Nationals
2796,WAS,Washington Nationals
338,WAS,Washington Senators
2564,WAS,Washington Nationals
2684,WAS,Washington Nationals
2744,WAS,Washington Nationals


In [131]:
team

,year,league_id,team_id,franchise_id,div_id,rank,g,w,l,div_win,wc_win,lg_win,ws_win,r,ab,h,double,triple,hr,bb,so,sb,cs,hbp,sf,era,sho,sv,ipouts,e
50,1876,NL,BSN,ATL,NaN,4,70,39,31,NaN,NaN,N,NaN,471,2722,723,96,24,9,58,98.0,NaN,NaN,NaN,NaN,2.51,3,7,1896,422
51,1876,NL,CHN,CHC,NaN,1,66,52,14,NaN,NaN,Y,NaN,624,2748,926,131,32,8,70,45.0,NaN,NaN,NaN,NaN,1.76,9,4,1776,282
52,1876,NL,CN1,CNR,NaN,8,65,9,56,NaN,NaN,N,NaN,238,2372,555,51,12,4,41,136.0,NaN,NaN,NaN,NaN,3.62,0,0,1773,469
53,1876,NL,HAR,HAR,NaN,2,69,47,21,NaN,NaN,N,NaN,429,2664,711,96,22,2,39,78.0,NaN,NaN,NaN,NaN,1.67,11,0,1872,337
54,1876,NL,LS1,LGR,NaN,5,69,30,36,NaN,NaN,N,NaN,280,2570,641,68,14,6,24,98.0,NaN,NaN,NaN,NaN,1.69,5,0,1929,396
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2800,2015,NL,LAN,LAD,W,1,162,92,70,Y,N,N,N,667,5385,1346,263,26,187,563,1258.0,59.0,34.0,60.0,30.0,3.44,21,47,4337,75
2801,2015,NL,SFN,SFG,W,2,162,84,78,N,N,N,N,696,5565,1486,288,39,136,457,1159.0,93.0,36.0,49.0,37.0,3.72,18,41,4333,78
2802,2015,NL,ARI,ARI,W,3,162,79,83,N,N,N,N,720,5649,1494,289,48,154,490,1312.0,132.0,44.0,33.0,57.0,4.04,12,44,4400,86
2803,2015,NL,SDN,SDP,W,4,162,74,88,N,N,N,N,650,5457,1324,260,36,148,426,1327.0,82.0,29.0,40.0,42.0,4.09,6,41,4321,92


In [170]:
salary.loc[salary['player_id'] == 'zimmejo02']

,year,team_id,league_id,player_id,salary,cpi,adj_salary
22291,2011,WAS,NL,zimmejo02,415000,224.939167,4.372829e+05
23139,2012,WAS,NL,zimmejo02,2300000,229.593917,2.374362e+06
23954,2013,WAS,NL,zimmejo02,5350000,232.957083,5.443238e+06
24756,2014,WAS,NL,zimmejo02,7500000,236.736167,7.508897e+06
25573,2015,WAS,NL,zimmejo02,16500000,237.017000,1.650000e+07


In [133]:
player

,player_id,birth_year,name_first,name_last,weight,height,bats,throws,bmi
0,aardsda01,1981.0,David,Aardsma,100.0,1.90,R,R,27.7
1,aaronha01,1934.0,Hank,Aaron,82.0,1.83,R,R,24.5
2,aaronto01,1939.0,Tommie,Aaron,86.0,1.90,R,R,23.8
3,aasedo01,1954.0,Don,Aase,86.0,1.90,R,R,23.8
4,abadan01,1972.0,Andy,Abad,83.0,1.85,L,L,24.3
...,...,...,...,...,...,...,...,...,...
18841,zupofr01,1939.0,Frank,Zupo,83.0,1.80,L,R,25.6
18842,zuvelpa01,1958.0,Paul,Zuvella,78.0,1.83,R,R,23.3
18843,zuverge01,1924.0,George,Zuverink,88.0,1.93,R,R,23.6
18844,zwilldu01,1888.0,Dutch,Zwilling,73.0,1.68,L,L,25.9


In [134]:
batting = pd.merge(batting, salary[['year', 'player_id', 'team_id', 'adj_salary']], on=['year', 'player_id', 'team_id'], how='left')
fielding = pd.merge(fielding, salary[['year', 'player_id', 'team_id', 'adj_salary']], on=['year', 'player_id', 'team_id'], how='left')
pitching = pd.merge(pitching, salary[['year', 'player_id', 'team_id', 'adj_salary']], on=['year', 'player_id', 'team_id'], how='left')
batting = pd.merge(batting, player[['player_id', 'weight', 'height', 'bmi', 'bats', 'throws', 'birth_year']], on=['player_id'], how='left')
fielding = pd.merge(fielding, player[['player_id', 'weight', 'height', 'bmi', 'bats', 'throws', 'birth_year']], on=['player_id'], how='left')
pitching = pd.merge(pitching, player[['player_id', 'weight', 'height', 'bmi', 'bats', 'throws', 'birth_year']], on=['player_id'], how='left')

In [169]:
batting['age'] = batting['year'] - batting['birth_year']
batting

,player_id,year,stint,team_id,league_id,g,ab,r,h,double,triple,hr,rbi,sb,cs,bb,so,hbp,sh,sf,adj_salary,weight,height,bmi,bats,throws,birth_year,age
0,addybo01,1876,1,CHN,NL,32,142.0,36.0,40.0,4.0,1.0,0.0,16.0,NaN,NaN,5.0,0.0,NaN,NaN,NaN,NaN,73.0,1.73,24.4,L,L,1842.0,34.0
1,allisar01,1876,1,LS1,NL,31,130.0,9.0,27.0,2.0,1.0,0.0,10.0,NaN,NaN,2.0,6.0,NaN,NaN,NaN,NaN,68.0,1.73,22.7,NaN,NaN,1849.0,27.0
2,allisdo01,1876,1,HAR,NL,44,163.0,19.0,43.0,4.0,0.0,0.0,15.0,NaN,NaN,3.0,9.0,NaN,NaN,NaN,NaN,73.0,1.78,23.0,R,R,1846.0,30.0
3,andrufr01,1876,1,CHN,NL,8,36.0,6.0,11.0,3.0,0.0,0.0,2.0,NaN,NaN,0.0,5.0,NaN,NaN,NaN,NaN,84.0,1.88,23.8,R,R,1850.0,26.0
4,ansonca01,1876,1,CHN,NL,66,309.0,63.0,110.0,9.0,7.0,2.0,59.0,NaN,NaN,12.0,8.0,NaN,NaN,NaN,NaN,103.0,1.83,30.8,R,R,1852.0,24.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97751,zitoba01,2015,1,OAK,AL,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,93.0,1.88,26.3,L,L,1978.0,37.0
97752,zobribe01,2015,1,OAK,AL,67,235.0,39.0,63.0,20.0,2.0,6.0,33.0,1.0,1.0,33.0,26.0,0.0,0.0,3.0,7500000.0,95.0,1.90,26.3,B,R,1981.0,34.0
97753,zobribe01,2015,2,KCA,AL,59,232.0,37.0,66.0,16.0,1.0,7.0,23.0,2.0,3.0,29.0,30.0,1.0,0.0,2.0,NaN,95.0,1.90,26.3,B,R,1981.0,34.0
97754,zuninmi01,2015,1,SEA,AL,112,350.0,28.0,61.0,11.0,0.0,11.0,28.0,0.0,1.0,21.0,132.0,5.0,8.0,2.0,523500.0,100.0,1.88,28.3,R,R,1991.0,24.0


In [167]:
fielding

,player_id,year,stint,team_id,league_id,pos,g,po,a,e,dp,adj_salary,weight,height,bmi,bats,throws,birth_year
0,addybo01,1876,1,CHN,NL,OF,32,46.0,6.0,13.0,0.0,NaN,73.0,1.73,24.4,L,L,1842.0
1,allisar01,1876,1,LS1,NL,1B,8,90.0,1.0,4.0,2.0,NaN,68.0,1.73,22.7,NaN,NaN,1849.0
2,allisar01,1876,1,LS1,NL,OF,23,34.0,11.0,12.0,1.0,NaN,68.0,1.73,22.7,NaN,NaN,1849.0
3,allisdo01,1876,1,HAR,NL,C,40,201.0,43.0,33.0,2.0,NaN,73.0,1.78,23.0,R,R,1846.0
4,allisdo01,1876,1,HAR,NL,OF,6,5.0,0.0,1.0,0.0,NaN,73.0,1.78,23.0,R,R,1846.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163711,younger03,2015,1,ATL,NL,OF,22,45.0,0.0,0.0,0.0,1000000.0,88.0,1.78,27.8,B,R,1985.0
163712,younger03,2015,2,NYN,NL,OF,7,5.0,0.0,0.0,0.0,NaN,88.0,1.78,27.8,B,R,1985.0
163713,zimmery01,2015,1,WAS,NL,OF,1,0.0,0.0,0.0,0.0,14000000.0,100.0,1.90,27.7,R,R,1984.0
163714,zobribe01,2015,1,OAK,AL,OF,29,39.0,0.0,3.0,0.0,7500000.0,95.0,1.90,26.3,B,R,1981.0


In [168]:
pitching

,player_id,year,stint,team_id,league_id,w,l,g,sho,sv,ipouts,h,er,hr,bb,so,era,hbp,bk,bfp,adj_salary,weight,height,bmi,bats,throws,birth_year
0,barnero01,1876,1,CHN,NL,0,0,1,0,0,4.0,7,3,0,0,0,20.25,NaN,0,13.0,NaN,66.0,1.73,22.1,R,R,1850.0
1,blongjo01,1876,1,SL3,NL,0,0,1,0,0,12.0,2,0,0,1,0,0.00,NaN,0,14.0,NaN,NaN,NaN,NaN,R,R,1853.0
2,bondto01,1876,1,HAR,NL,31,13,45,6,0,1224.0,355,76,2,13,88,1.68,NaN,0,1623.0,NaN,73.0,1.70,25.3,R,R,1856.0
3,bootham01,1876,1,CN1,NL,0,1,3,0,0,29.0,22,10,0,0,0,9.31,NaN,0,51.0,NaN,72.0,1.75,23.5,R,R,1848.0
4,boothed01,1876,1,NY3,NL,0,0,1,0,0,15.0,16,6,0,0,0,10.80,NaN,0,34.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43020,youngch03,2015,1,KCA,AL,11,6,34,0,0,370.0,91,42,16,43,83,3.06,0.0,0,500.0,675000.0,116.0,2.08,26.8,R,R,1979.0
43021,zieglbr01,2015,1,ARI,NL,0,3,66,0,30,204.0,48,14,3,17,36,1.85,1.0,0,263.0,5000000.0,100.0,1.93,26.8,R,R,1979.0
43022,zimmejo02,2015,1,WAS,NL,13,10,33,0,0,605.0,204,82,24,39,164,3.66,8.0,1,831.0,16500000.0,102.0,1.88,28.9,R,R,1986.0
43023,zitoba01,2015,1,OAK,AL,0,0,3,0,0,21.0,12,8,4,6,2,10.29,0.0,0,37.0,NaN,93.0,1.88,26.3,L,L,1978.0
